In [6]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

from dotenv import load_dotenv

from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [7]:
load_dotenv()

True

In [8]:
model_emb = OpenAIEmbeddings(
    model = 'text-embedding-3-large',
    dimensions = 32
)

model_gemini = GoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)

In [9]:
doc1 = Document(page_content = (
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""),
    metadata = {"source": "Doc1"}, id = '1')

doc2 = Document(
    page_content =
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls.""",
    metadata = {"source": "Doc2"}, id = '2')

doc3 = Document(page_content = 
                ("""Basketball was invented by Dr. James Naismith in the late 19th century.
                It was originally played with a soccer ball and peach baskets. NBA is now a global league."""),
                metadata = {"source": "Doc3"}, id = '3')

doc4 = Document(page_content =
                ("""The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""),
                metadata = {"source": "Doc4"}, id = '4')


docs = [doc1, doc2, doc3, doc4]

In [14]:
# Initializing vector store
vec_store = Chroma.from_documents(
    documents = docs,
    embedding = model_emb,
    collection_name = "Sample_docs",
    persist_directory = "CCR",
)

In [15]:
# Initializing retriever
retriever = vec_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = {'k': 2, 'lambda_mult': 0.5}
)

In [16]:
# initializing compresssor

compressor = LLMChainExtractor.from_llm(
    llm = model_gemini
)

# Initializing contextua; compression retriever
ccr = ContextualCompressionRetriever(
    base_retriever = retriever,
    base_compressor = compressor
)

In [17]:
# Passing the query

result = ccr.invoke("What is Photosynthesis?")

In [18]:
result

[Document(metadata={'source': 'Doc1'}, page_content='Photosynthesis is the process by which green plants convert sunlight into energy.'),
 Document(metadata={'source': 'Doc2'}, page_content='The chlorophyll in plant cells captures sunlight during photosynthesis.')]

In [29]:
for i, doc in enumerate(result):
    print(f'''
---Result {i+1}---
{doc.page_content}''')

    print()
    print()


---Result 1---
Photosynthesis is the process by which green plants convert sunlight into energy.



---Result 2---
The chlorophyll in plant cells captures sunlight during photosynthesis.




In [30]:
compressor

LLMChainExtractor(llm_chain=PromptTemplate(input_variables=['context', 'question'], input_types={}, output_parser=NoOutputParser(), partial_variables={}, template='Given the following question and context, extract any part of the context *AS IS* that is relevant to answer the question. If none of the context is relevant return NO_OUTPUT.\n\nRemember, *DO NOT* edit the extracted parts of the context.\n\n> Question: {question}\n> Context:\n>>>\n{context}\n>>>\nExtracted relevant parts:')
| GoogleGenerativeAI(model='gemini-2.5-flash', google_api_key=SecretStr('**********'), client=genai.GenerativeModel(
      model_name='models/gemini-2.5-flash',
      generation_config={},
      safety_settings={},
      tools=None,
      system_instruction=None,
      cached_content=None
  ))
| NoOutputParser(), get_input=<function default_get_input at 0x00000269C7FD6E80>)

In [31]:
ccr

ContextualCompressionRetriever(base_compressor=LLMChainExtractor(llm_chain=PromptTemplate(input_variables=['context', 'question'], input_types={}, output_parser=NoOutputParser(), partial_variables={}, template='Given the following question and context, extract any part of the context *AS IS* that is relevant to answer the question. If none of the context is relevant return NO_OUTPUT.\n\nRemember, *DO NOT* edit the extracted parts of the context.\n\n> Question: {question}\n> Context:\n>>>\n{context}\n>>>\nExtracted relevant parts:')
| GoogleGenerativeAI(model='gemini-2.5-flash', google_api_key=SecretStr('**********'), client=genai.GenerativeModel(
      model_name='models/gemini-2.5-flash',
      generation_config={},
      safety_settings={},
      tools=None,
      system_instruction=None,
      cached_content=None
  ))
| NoOutputParser(), get_input=<function default_get_input at 0x00000269C7FD6E80>), base_retriever=VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore

# End of the topic.
# Go to word file for the last time..